In [1]:
# ============================================================
# ERIP - SILVER CUSTOMER DIMENSION
# Notebook      : nb_build_customer_dimension
# Layer         : Silver
#
# Purpose
# --------
# Build the conformed Customer Dimension from the Bronze
# Customer Master table.
#
# Enterprise Concepts Demonstrated
# --------------------------------
# • Medallion Architecture (Bronze → Silver)
# • Data Standardization
# • Master Data Management (MDM)
# • Conformed Dimensions
# • Analytics Engineering
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

source_table = "bronze_customer_master"
target_table = "silver_customer"
pipeline_name = "nb_build_customer_dimension"

run_start_time = datetime.now()

print("ERIP Silver Customer Dimension Build Started")

StatementMeta(, e933cd3c-027c-4d45-af50-b4c1dacace38, 3, Finished, Available, Finished, False)

ERIP Silver Customer Dimension Build Started


In [2]:
# ============================================================
# SECTION 2 - READ BRONZE TABLE
#
# Purpose
# -------
# Read validated/governed Bronze Delta table from Lakehouse.
# ============================================================

customer_bronze_df = spark.table(source_table)

display(customer_bronze_df.limit(10))

print(f"Rows read: {customer_bronze_df.count()}")
print(f"Columns read: {len(customer_bronze_df.columns)}")

StatementMeta(, e933cd3c-027c-4d45-af50-b4c1dacace38, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1a95a665-b85a-4376-ac94-a5fe484d142d)

Rows read: 1000
Columns read: 29


In [3]:
# ============================================================
# SECTION 3 - CUSTOMER STANDARDIZATION
#
# Purpose
# -------
# Standardize customer attributes to create a clean,
# enterprise-ready business entity.
#
# Transformations
# ---------------
# • Trim whitespace
# • Standardize text case
# • Convert data types
# • Standardize dates
# • Remove duplicate customers
#
# Enterprise Concepts
# -------------------
# • Data Standardization
# • Master Data Management
# • Data Conformance
# ============================================================

silver_customer_df = (
    customer_bronze_df
    .select(
        col("customer_id"),
        initcap(trim(col("customer_name"))).alias("customer_name"),
        upper(trim(col("legal_entity_id"))).alias("legal_entity_id"),
        upper(trim(col("lei_code"))).alias("lei_code"),
        initcap(trim(col("legal_entity_type"))).alias("legal_entity_type"),
        upper(trim(col("customer_group_id"))).alias("customer_group_id"),
        initcap(trim(col("parent_company_name"))).alias("parent_company_name"),
        upper(trim(col("industry_code"))).alias("industry_code"),
        initcap(trim(col("industry_name"))).alias("industry_name"),
        upper(trim(col("nace_code"))).alias("nace_code"),
        upper(trim(col("country_code"))).alias("country_code"),
        initcap(trim(col("country"))).alias("country"),
        initcap(trim(col("region"))).alias("region"),
        initcap(trim(col("risk_country"))).alias("risk_country"),
        initcap(trim(col("segment"))).alias("segment"),
        col("annual_revenue").cast("double").alias("annual_revenue"),
        col("total_assets").cast("double").alias("total_assets"),
        initcap(trim(col("kyc_risk_rating"))).alias("kyc_risk_rating"),
        col("esg_score").cast("int").alias("esg_score"),
        initcap(trim(col("esg_risk_band"))).alias("esg_risk_band"),
        to_date(col("relationship_start_date")).alias("relationship_start_date"),
        to_date(col("onboarding_date")).alias("onboarding_date"),
        upper(trim(col("relationship_manager"))).alias("relationship_manager"),
        initcap(trim(col("customer_status"))).alias("customer_status"),
        current_timestamp().alias("silver_updated_timestamp")
    )
    .dropDuplicates(["customer_id"])
)

StatementMeta(, e933cd3c-027c-4d45-af50-b4c1dacace38, 5, Finished, Available, Finished, False)

In [4]:
# ============================================================
# SECTION 4 - BUSINESS ENRICHMENT
#
# Purpose
# -------
# Enrich the standardized customer data with derived business
# attributes used for enterprise analytics.
#
# Derived Attributes
# ------------------
# • Customer Surrogate Key
# • Customer Tenure
# • Enterprise Revenue Band
# • Customer Risk Category
#
# Enterprise Concepts
# -------------------
# • Surrogate Keys
# • Dimensional Modeling
# • Business Rules
# • Customer Segmentation
# • Customer 360 Foundation
# ============================================================

window_spec = Window.orderBy("customer_id")

silver_customer_df = (
    silver_customer_df
    .withColumn("customer_sk", row_number().over(window_spec))
    .withColumn(
        "customer_tenure_years",
        floor(months_between(current_date(), col("relationship_start_date")) / 12)
    )
    .withColumn(
        "enterprise_revenue_band",
        when(col("annual_revenue") >= 1000000000, "Tier 1 - Strategic")
        .when(col("annual_revenue") >= 250000000, "Tier 2 - Large Corporate")
        .when(col("annual_revenue") >= 50000000, "Tier 3 - Mid Market")
        .otherwise("Tier 4 - Commercial")
    )
    .withColumn(
        "customer_risk_category",
        when((col("kyc_risk_rating") == "High") | (col("esg_score") < 40), "High Risk")
        .when((col("kyc_risk_rating") == "Medium") | (col("esg_score") < 60), "Medium Risk")
        .otherwise("Low Risk")
    )
    .select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "legal_entity_id",
        "lei_code",
        "legal_entity_type",
        "customer_group_id",
        "parent_company_name",
        "industry_code",
        "industry_name",
        "nace_code",
        "country_code",
        "country",
        "region",
        "risk_country",
        "segment",
        "annual_revenue",
        "total_assets",
        "enterprise_revenue_band",
        "kyc_risk_rating",
        "esg_score",
        "esg_risk_band",
        "customer_risk_category",
        "relationship_start_date",
        "customer_tenure_years",
        "onboarding_date",
        "relationship_manager",
        "customer_status",
        "silver_updated_timestamp"
    )
)

display(silver_customer_df.limit(10))

StatementMeta(, e933cd3c-027c-4d45-af50-b4c1dacace38, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 871acaee-862b-43b2-bc81-084feca8d9fb)

In [5]:
# ============================================================
# SECTION 5 - SILVER CUSTOMER QUALITY CHECKS
# ============================================================

total_rows = silver_customer_df.count()
duplicate_customer_ids = total_rows - silver_customer_df.select("customer_id").distinct().count()
null_customer_ids = silver_customer_df.filter(col("customer_id").isNull()).count()
null_surrogate_keys = silver_customer_df.filter(col("customer_sk").isNull()).count()

print("Silver Customer Quality Checks")
print("------------------------------")
print(f"Rows: {total_rows}")
print(f"Duplicate customer IDs: {duplicate_customer_ids}")
print(f"Null customer IDs: {null_customer_ids}")
print(f"Null surrogate keys: {null_surrogate_keys}")

if duplicate_customer_ids > 0 or null_customer_ids > 0 or null_surrogate_keys > 0:
    raise Exception("Silver customer quality validation failed")
else:
    print("✓ Silver customer validation passed")

StatementMeta(, e933cd3c-027c-4d45-af50-b4c1dacace38, 7, Finished, Available, Finished, False)

Silver Customer Quality Checks
------------------------------
Rows: 1000
Duplicate customer IDs: 0
Null customer IDs: 0
Null surrogate keys: 0
✓ Silver customer validation passed


In [6]:
# ============================================================
# SECTION 6 - WRITE SILVER CUSTOMER DELTA TABLE
# ============================================================

silver_customer_df.write.mode("overwrite").format("delta").saveAsTable(target_table)

print(f"✓ Silver table created: {target_table}")
print(f"Rows written: {silver_customer_df.count()}")

StatementMeta(, e933cd3c-027c-4d45-af50-b4c1dacace38, 8, Finished, Available, Finished, False)

✓ Silver table created: silver_customer
Rows written: 1000
